In [14]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = (
    SparkSession.builder.appName("ExecutionPlanApp")
    .config("spark.executor.memory", "1g")
    .getOrCreate()
)

# 2. Create 10 rows with bad column names & realistic edge cases (nulls, negative qty)
raw_data = [
    (1, "Laptop", "Electronics", 1200.0, 1, "Completed"),
    (2, "Desk Chair", "Furniture", 250.0, 2, "Completed"),
    (3, "Wireless Mouse", "Electronics", 35.0, 4, "Completed"),
    (4, "Monitor", "Electronics", None, 1, "Completed"),  # Null price
    (5, "Standing Desk", "Furniture", 550.0, 1, "Completed"),
    (6, "Webcam", "Electronics", 85.0, -2, "Completed"),  # Invalid negative quantity
    (7, "Notebook Pack", "Stationery", 12.0, 10, "Cancelled"),  # Cancelled order
    (8, "Desk Lamp", "Furniture", 45.0, 3, "Completed"),
    (9, "USB-C Hub", "Electronics", 60.0, 2, "Completed"),
    (10, "Pen Set", "Stationery", 15.0, 5, "Completed"),
]

# Intentional bad column names (spaces, capital letters, symbols)
bad_columns = [
    "Order ID",
    "Product Name",
    "Category",
    "Unit Price ($)",
    "Qty",
    "Order_Status",
]

df = spark.createDataFrame(raw_data, schema=bad_columns)

df.show(20)

[Stage 0:>                                                          (0 + 1) / 1]

+--------+--------------+-----------+--------------+---+------------+
|Order ID|  Product Name|   Category|Unit Price ($)|Qty|Order_Status|
+--------+--------------+-----------+--------------+---+------------+
|       1|        Laptop|Electronics|        1200.0|  1|   Completed|
|       2|    Desk Chair|  Furniture|         250.0|  2|   Completed|
|       3|Wireless Mouse|Electronics|          35.0|  4|   Completed|
|       4|       Monitor|Electronics|          NULL|  1|   Completed|
|       5| Standing Desk|  Furniture|         550.0|  1|   Completed|
|       6|        Webcam|Electronics|          85.0| -2|   Completed|
|       7| Notebook Pack| Stationery|          12.0| 10|   Cancelled|
|       8|     Desk Lamp|  Furniture|          45.0|  3|   Completed|
|       9|     USB-C Hub|Electronics|          60.0|  2|   Completed|
|      10|       Pen Set| Stationery|          15.0|  5|   Completed|
+--------+--------------+-----------+--------------+---+------------+



### SPARK EXECUTION PLAN 

The below script is for us to understand how the `Spark Driver` use the `Catalyst Optmizer` to carry out execution strategy on our spark application. What we are doing in the code is simple, we are 
- Filtering out where `Order_Status == completed` and the `Qty` column is `greater than 0`.
- We do `aggregation` that let us run a `total of Qty accros Categories`, which we now rename that total column as `total_qty`.
- We now select `Category` and the `total_qty` 

In [15]:
# Step 1: Filter out cancelled orders and invalid (negative) quantities
filtered_df = df.filter((F.col("Order_Status") == "Completed") & (F.col("Qty") > 0))

# Step 2: Group the surviving rows by Category
grouped_df = filtered_df.groupBy("Category")

# Step 3: Aggregate (sum) the Qty for each group and name the new column
aggregated_df = grouped_df.agg(F.sum("Qty").alias("total_qty"))

# Step 4: Explicitly select the final 2 columns
result_df = aggregated_df.select("Category", "total_qty")

# Inspect the execution plan and trigger the action
result_df.explain(mode="extended")


== Parsed Logical Plan ==
'Project ['Category, 'total_qty]
+- Aggregate [Category#100], [Category#100, sum(Qty#102L) AS total_qty#123L]
   +- Filter ((Order_Status#103 = Completed) AND (Qty#102L > cast(0 as bigint)))
      +- LogicalRDD [Order ID#98L, Product Name#99, Category#100, Unit Price ($)#101, Qty#102L, Order_Status#103], false

== Analyzed Logical Plan ==
Category: string, total_qty: bigint
Project [Category#100, total_qty#123L]
+- Aggregate [Category#100], [Category#100, sum(Qty#102L) AS total_qty#123L]
   +- Filter ((Order_Status#103 = Completed) AND (Qty#102L > cast(0 as bigint)))
      +- LogicalRDD [Order ID#98L, Product Name#99, Category#100, Unit Price ($)#101, Qty#102L, Order_Status#103], false

== Optimized Logical Plan ==
Aggregate [Category#100], [Category#100, sum(Qty#102L) AS total_qty#123L]
+- Project [Category#100, Qty#102L]
   +- Filter ((isnotnull(Order_Status#103) AND isnotnull(Qty#102L)) AND ((Order_Status#103 = Completed) AND (Qty#102L > 0)))
      +- Logic

### Execution Plan Summary

Reading from Top to Bottom

- `Parsed Logical Plan`: This is the phase where your intent is read out by the `Spark Driver`, no checks are done at this point if your columns doesn't exist, if there is a typo.
  - Reading that from `bottom to Top`, you will see it matches exactly the code you wrote, `filtering`, `aggregate` and later `select` your interested columns.
  
- `Analyzed Logical Plan`: This is the phase where the Catalyst Optimzer inspect your code, columns referenced e.t.c to see if they exist, if the column is not available for example, you get an `AnalysisException: UNRESOLVED COLUMN` error here.
    - You can see your intent , the order of the opertaion in your code is stil exactly the same.
  
- `Optimized Logical Plan`: This pahse is where the optimization starts, the Catalyst Optimizer can see that you are doing an aggregation before selecting Category and total_qty.
    - It saw that you are not using some other columns at all in the whole operation.
    - It decided to do the selection of the 2 columns before the aggregation instead.
    
- `Physical Plan`: After Optimization, the physical plan is the final plan of how the work will be computed on the Executor
    - Scanning of the data
    - Filtering
    - Select the 2 columns
    - Aggregation afterwards
    - Notice the Exchange ? That's an indication that shuffle will be triggered, which is expected due to wide transformation.

In [13]:
spark.stop()